# 🚀 V-Netra: YOLO11n Training on Colab (Manual/Pre-built Dataset)
Notebook ini khusus digunakan jika Anda **sudah memiliki dataset ZIP** di Google Drive Anda. Proses training akan langsung dijalankan tanpa perlu mengunduh ulang dari Roboflow.


In [ ]:
!pip install -q albumentations

from google.colab import drive
import os
import shutil
import zipfile
drive.mount('/content/drive')

import os

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml fiftyone Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")

## 1. Ekstrak Dataset


In [ ]:
import os
import shutil
import zipfile

# 3. Ekstrak Dataset
master_dir = "/content/vnetra_master_dataset"
vnetra_master_dataset_zip_path = f'{INPUT_DIR}/vnetra_master_dataset.zip'

if not os.path.exists(master_dir):
    if os.path.exists(vnetra_master_dataset_zip_path):
        print("Mengekstrak vnetra master dataset dari Google Drive...")
        os.makedirs(master_dir, exist_ok=True)
        shutil.unpack_archive(vnetra_master_dataset_zip_path, master_dir)
        print("✅ vnetra master dataset siap digunakan.")
    else:
        print("❌ ERROR: vnetra_master_dataset.zip tidak ditemukan. Silakan jalankan Fase 1 terlebih dahulu.")
else:
    print("✅ vnetra master dataset sudah tersedia di memori.")

## 2. Pengecekan Statistik Dataset


In [ ]:
master_classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'pole',
    'tactile_paving_straight', 'tactile_paving_turn', 
    'tactile_paving_3way', 'tactile_paving_4way', 'tactile_paving_stop',
    'stairs_up', 'stairs_down', 'crosswalk', 'tree'
]

import pandas as pd
import os

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test
    
    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total
    
    data_report.append({
        'ID': i, 
        'Kelas': cls_name, 
        'Train (Inst)': t_train, 
        'Valid (Inst)': t_valid, 
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-', 
    'Kelas': 'TOTAL KESELURUHAN', 
    'Train (Inst)': total_train, 
    'Valid (Inst)': total_valid, 
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


## 3. Training Model YOLO11n


In [ ]:
from ultralytics import YOLO

# Memuat arsitektur dasar YOLO11 versi nano (Paling ringan dan cepat untuk mobile)
model = YOLO('yolo11n.pt')

results = model.train(
    # --- KONFIGURASI DATA & PERANGKAT ---
    data=f"{master_dir}/data.yaml", # Path menuju dataset yang sudah digabung
    epochs=300,                     # Maksimal putaran training (300 sudah lebih dari cukup)
    time=11.0,                      # Otomatis Berhenti & Save dengan aman setelah 11 jam (Mencegah Colab mendadak mati)
    patience=50,                    # Jika dalam 50 epoch akurasi tidak naik, hentikan training lebih awal (Early Stopping)
    imgsz=640,                      # Resolusi standar YOLO (kamera OV2640 akan di-resize ke ukuran ini)
    batch=32,                       # Memproses 32 gambar sekaligus (menyesuaikan kapasitas RAM GPU T4 Colab)
    device=0,                       # Menggunakan GPU ke-0 (Wajib menggunakan GPU untuk YOLO)
    workers=4,                      # Menggunakan 4 core CPU untuk memuat gambar ke GPU lebih cepat
    seed=42,                        # Angka acak tetap agar hasil training bisa direproduksi/konsisten
    
    # --- PENYIMPANAN LOG & GRAFIK ---
    project='vnetra_training',      # Nama folder utama penyimpanan hasil
    name='yolo11n_custom',          # Nama sub-folder spesifik untuk eksperimen ini
    exist_ok=True,                  # Menimpa folder jika sudah ada (mencegah penumpukan folder eksperimen)
    save_period=10,                 # Menyimpan file bobot cadangan setiap 10 putaran
    
    # --- STRATEGI PEMBELAJARAN (LEARNING) ---
    freeze=5,                       # Membekukan (tidak melatih ulang) 5 layer awal yang sudah mahir mendeteksi tepi benda (menghemat waktu)
    lr0=0.002,                      # Kecepatan belajar awal (tidak terlalu besar agar tidak 'nyasar', tidak terlalu kecil agar tidak lambat)
    cos_lr=True,                    # Menurunkan kecepatan belajar secara perlahan membentuk kurva kosinus (memuluskan akurasi di akhir)
    warmup_epochs=1.0,              # Pemanasan 1 epoch pertama dengan kecepatan sangat rendah agar model tidak kaget
    
    # --- AUGMENTASI KHUSUS VNETRA (OV2640 CAMERA SIMULATION) ---
    mosaic=1.0,                     # Menggabungkan 4 gambar jadi 1, melatih model mendeteksi objek kecil dalam satu frame
    degrees=20.0,                   # Rotasi lebih natural (10 derajat) untuk kamera kepala/kacamata tanpa merusak bentuk objek
    fliplr=0.0,                     # DIMATIKAN! Jangan membalik gambar kiri-kanan, karena arah Tactile Paving (belok kiri vs kanan) bisa tertukar
    scale=0.5,                      # Skala zooming natural (YOLO default) agar objek tetap utuh
    
    # Simulasi kualitas gambar buruk dari kamera OV2640 (warna pudar, gelap, dll)
    hsv_h=0.015,                    # Fluktuasi hue (warna dasar)
    hsv_s=0.5,                      # Fluktuasi saturation moderat (50%) agar warna tetap wajar
    hsv_v=0.4,                      # Fluktuasi kecerahan (value) moderat (40%) meniru bayangan natural
    erasing=0.1,                    # Menghapus porsi gambar secara acak diturunkan ke 10%
)


## 4. Ekspor Model dan Backup ke Google Drive


In [ ]:
import shutil
import os

# Pindahkan model .pt asli
best_pt_path = '/content/vnetra_training/yolo11n_custom/weights/best.pt'
if os.path.exists(best_pt_path):
    shutil.copy(best_pt_path, f'{OUTPUT_DIR}/best_yolo11n.pt')
    print("✅ Model Asli (.pt) berhasil disimpan ke Google Drive!")

print("\nMengekspor model ke FP16 (TFLite)...")
export_fp16 = model.export(format="tflite", half=True, optimize=True)

if os.path.exists(export_fp16):
    shutil.copy(export_fp16, f'{OUTPUT_DIR}/best_fp16.tflite')
    print('✅ Model FP16 berhasil disimpan ke Google Drive!')


## 5. Evaluasi pada Test Set


In [ ]:
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) PADA TEST SET ===")
val_pt = model.val(data=f"{master_dir}/data.yaml", split='test')
map_pt = val_pt.box.map50

print("\n=== EVALUASI MODEL FP16 (.tflite) PADA TEST SET ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml", split='test')
map_fp16 = val_fp16.box.map50

print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 PADA DATASET TEST:")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")
print("=========================================")


## 6. Uji Coba Visual


In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import glob

test_images = glob.glob(f"{master_dir}/test/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji gambar: {test_img}")
    
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    res_fp16 = model_fp16.predict(source=test_img, imgsz=640)
    img_fp16 = res_fp16[0].plot()
    
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"Original YOLO11n (.pt)\nConfidence, mAP: {map_pt:.2f}")
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_fp16, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f"TFLite FP16 (.tflite)\nConfidence, mAP: {map_fp16:.2f}")
    ax[1].axis("off")
    plt.show()
else:
    print("Folder test kosong!")


## 7. Dashboard Training


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

base_path = '/content/vnetra_training/yolo11n_custom/'
results_path = os.path.join(base_path, 'results.csv')

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip() 
    
    sns.set_theme(style='whitegrid', palette='deep')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.lineplot(ax=axes[0], data=df, x='epoch', y='train/box_loss', label='Train Box Loss', lw=2)
    sns.lineplot(ax=axes[0], data=df, x='epoch', y='val/box_loss', label='Val Box Loss', lw=2, linestyle='--')
    axes[0].set_title('Penurunan Box Loss (Semakin kecil semakin baik)', fontsize=14, fontweight='bold')
    
    sns.lineplot(ax=axes[1], data=df, x='epoch', y='metrics/mAP50(B)', label='mAP@50', lw=2, color='green')
    sns.lineplot(ax=axes[1], data=df, x='epoch', y='metrics/mAP50-95(B)', label='mAP@50-95', lw=2, color='teal', linestyle='--')
    axes[1].set_title('Peningkatan Akurasi mAP (Semakin tinggi semakin baik)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

    print("\n=== GRAFIK BAWAAN YOLO (LENGKAP) ===")
    img_files = ['results.png', 'confusion_matrix.png', 'PR_curve.png']
    for img in img_files:
        img_path = os.path.join(base_path, img)
        if os.path.exists(img_path):
            print(f"\nMenampilkan: {img}")
            display(Image(filename=img_path))
else:
    print("File results.csv tidak ditemukan. Training mungkin gagal atau path salah.")


## 8. Backup Seluruh Hasil Training & Bersihkan


In [ ]:
import os
import shutil

print(f"\nMenge-ZIP dan membackup seluruh hasil training ke {OUTPUT_DIR}...")
shutil.make_archive(f"{OUTPUT_DIR}/vnetra_training_results", 'zip', "/content/vnetra_training")

print(f"\n✅ Backup selesai! Semua hasil tersimpan di {OUTPUT_DIR}/vnetra_training_results.zip")

print("\n🗑️ Membersihkan dataset dari memori lokal Colab...")
if os.path.exists(master_dir):
    shutil.rmtree(master_dir)
print("✅ Pembersihan selesai.")
